<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week01_gd_optimization/gd_capstone.ipynb)

# Gradient-Based Optimization: From Calculus to GD/SGD

**TAE Week 1 Capstone (Core Track)** · Francisco (FranQuant)

Calculus analysis of a 1-D piecewise-smooth objective, then GD/SGD experiments per the
handout's Section 7 protocol and Capstone Delivery Checklist.

## 0. Scope, References, and Reproducibility Notes

**Objective under study**

$$
f(x) \;=\; \left|\tfrac{1}{2}x^{3} - \tfrac{3}{2}x^{2}\right| + \tfrac{1}{2}x .
$$

**What this notebook delivers** (per the Capstone Delivery Checklist):

1. Definitions of $f$, $f'$ (piecewise), and a subgradient selection at the kink $x=3$.
2. An analytic-vs-numeric gradient check (acceptance rubric requirement).
3. GD experiments: 3 initializations × 4 constant step sizes, horizon $K=200$.
4. SGD experiments: constant vs diminishing step-size schedules.
5. Replications of the handout's key figures (function, derivative, step geometry,
   trajectories, overshoot, SGD paths, schedule comparison) with titles, axis labels,
   and captions.
6. Metrics: final gap $f(x_K)-f(x_\star)$, best-so-far gap, steps-to-tolerance;
   paired seed statistics and kink-basin escape fractions for SGD.
7. A quadratic stability baseline (coach guide scope) and an empirical probe of the
   kink's limit-cycle trap.

**Honest-reporting note.** Figures 6–9 and 11 in the handout are explicitly *schematic*
("illustrative rather than the output of a fixed numerical run"). All figures in this
notebook are **actual seeded runs**; qualitative agreement with the handout's schematics
is discussed in the interpretation notes.

**References.** [1] `gd_optimization.pdf` (case-study handout) · [2] PM Book, Chs. 8–10, 15, 17 ·
[3] `tae_week_1_coach.pdf`, §13 Capstone Brief & Acceptance Rubric · [4] Nesterov (2004) ·
[5] Robbins & Monro (1951).

## 1. Setup

All packages are in the standard Colab image; the cell below is a no-op on Colab and a
safety net elsewhere. Runtime target for the full notebook: **< 2 minutes**.

In [ ]:
# Install-if-missing (no-op on Colab; kept for the "single-click" requirement)
import importlib.util, subprocess, sys
for pkg in ("numpy", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:  # pragma: no cover
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


In [ ]:
%config InlineBackend.figure_format = 'retina'
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8')  # house style (matches pmcode conventions)

ASSETS = Path("assets")        # relative path only (checklist item 6)
ASSETS.mkdir(exist_ok=True)

def save_fig(fig, name):
    """Deterministic figure saving (fixed dpi, tight bbox)."""
    fig.savefig(ASSETS / f"{name}.png", dpi=300, bbox_inches="tight")


## 2. Hyperparameters & Seed (single source of truth)

All experiment knobs live here (checklist item 3). Values follow the handout's
Section 7 protocol exactly.

In [ ]:
SEED = 1                                    # fixed random seed
rng = np.random.default_rng(SEED)

# --- Protocol (handout, Section 7) ---
X0_LIST   = [-1.0, 0.5, 2.0]                # initializations
ETA_GD    = [0.05, 0.10, 0.15, 0.20]        # GD constant step sizes
ETA0_SGD  = [0.2, 0.1]                      # SGD initial steps (diminishing schedule)
GAMMA_SGD = [0.02, 0.05]                    # SGD decay rates
K         = 200                             # iteration horizon
SIGMA     = 0.5                             # SGD noise std (E[eps]=0, E[eps^2]<=sigma^2)
TOL       = 1e-3                            # tolerance for steps-to-tolerance metric

# --- Demo-specific values from the handout ---
ETA_GEOM      = 0.2    # step-geometry figure (Fig. 3): x0=2, eta=0.2 -> x'=1.9
ETA_TRAJ      = 0.15   # three-trajectory figure (Fig. 4)
ETA_OVERSHOOT = 0.6    # oscillation demo (Fig. 5): x0=0.5


## 3. Problem Setup: $f$, $f'$, Subgradient, Analytic Minimizer

Write $g(x) = \tfrac12 x^3 - \tfrac32 x^2 = \tfrac12 x^2 (x-3)$, so $f(x) = |g(x)| + \tfrac12 x$.

**Piecewise form** (handout, Prop. 1). Since $g(x)\le 0$ for $x<3$ and $g(x)>0$ for $x>3$:

$$
f(x) =
\begin{cases}
-\tfrac12 x^3 + \tfrac32 x^2 + \tfrac12 x, & x < 3,\\[2pt]
\;\;\tfrac12 x^3 - \tfrac32 x^2 + \tfrac12 x, & x > 3.
\end{cases}
$$

$f$ is differentiable at $x=0$ (because $g(0)=g'(0)=0$; handout Lemma 1) with $f'(0)=\tfrac12$,
and non-differentiable only at the kink $x=3$, where the one-sided limits are
$\lim_{x\uparrow 3} f'(x) = -4$ and $\lim_{x\downarrow 3} f'(x) = 5$, hence

$$
\partial f(3) = [-4,\, 5].
$$

**Derivative on each branch:**

$$
f'(x) =
\begin{cases}
-\tfrac32 x^2 + 3x + \tfrac12, & x < 3,\\[2pt]
\;\;\tfrac32 x^2 - 3x + \tfrac12, & x > 3.
\end{cases}
$$

**Stationary points & global minimizer** (handout, Thm. 1). On $(-\infty,3)$ the roots of
$f'$ are $x_\pm = 1 \mp \tfrac{2}{3}\sqrt{3}$; the second-derivative test gives
$x_- = 1-\tfrac{2}{3}\sqrt{3}\approx -0.1547$ as a strict local (and unique global) minimizer
and $x_+\approx 2.1547$ as a strict local maximizer. For $x>3$, $f'>0$. The global minimum value is

$$
f(x_\star) = \tfrac{3}{2} - \tfrac{8}{9}\sqrt{3} \;\approx\; -3.96\times 10^{-2}.
$$

In [ ]:
def f(x):
    """Objective f(x) = |0.5 x^3 - 1.5 x^2| + 0.5 x."""
    x = np.asarray(x, dtype=np.float64)
    return np.abs(0.5 * x**3 - 1.5 * x**2) + 0.5 * x

def fprime(x):
    """Piecewise derivative on smooth branches (defined for x != 3)."""
    x = np.asarray(x, dtype=np.float64)
    left  = -1.5 * x**2 + 3.0 * x + 0.5   # x < 3
    right =  1.5 * x**2 - 3.0 * x + 0.5   # x > 3
    return np.where(x < 3.0, left, right)

def subgrad(x):
    """Subgradient selection: f'(x) off the kink; smallest-norm element of
    [-4, 5] (i.e., 0) exactly at x = 3 (handout, Section 8 recommendation).
    Exact equality (not a tolerance band) so nearby smooth points keep their
    true derivative; x = 3 is then an exact fixed point of the update."""
    if x == 3.0:
        return 0.0                        # smallest-norm selection in [-4, 5]
    return float(fprime(x))

# Analytic constants (exact expressions)
X_STAR = 1.0 - (2.0 / 3.0) * np.sqrt(3.0)          # global minimizer
F_STAR = 3.0 / 2.0 - (8.0 / 9.0) * np.sqrt(3.0)    # global minimum value
X_PLUS = 1.0 + (2.0 / 3.0) * np.sqrt(3.0)          # local maximizer

print(f"x_star = {X_STAR:+.6f}   f(x_star) = {F_STAR:+.6f}")
print(f"consistency check: f(X_STAR) - F_STAR = {f(X_STAR) - F_STAR:+.2e}")
assert np.isclose(f(X_STAR), F_STAR), "analytic minimum value mismatch"


## 4. Gradient Check: Analytic vs Numeric

Central differences $\dfrac{f(x+h)-f(x-h)}{2h}$ against the analytic $f'$ on both smooth
branches, avoiding the kink. Expected agreement: $\mathcal{O}(h^2)$.

In [ ]:
h = 1e-6
xs_check = np.concatenate([
    np.linspace(-2.0, 2.8, 13),   # left branch (avoids kink)
    np.linspace(3.2, 4.0, 5),     # right branch
])
num  = (f(xs_check + h) - f(xs_check - h)) / (2.0 * h)
ana  = fprime(xs_check)
err  = np.abs(num - ana)
print(f"max |analytic - numeric| = {err.max():.3e}  (n = {len(xs_check)} points)")
assert err.max() < 1e-6, "gradient check failed"
print("gradient check PASSED")


## 5. Function and Derivative

**Sign pattern of $f'$:** negative on $(-\infty, x_-)$, positive on $(x_-, x_+)$,
**negative on $(x_+, 3)$**, positive on $(3, \infty)$.

**A fact worth stating explicitly (follows from the handout's own calculus):** since
$0 \in \partial f(3) = [-4, 5]$ and $f' < 0$ on $(x_+, 3)$ while $f' > 0$ on $(3,\infty)$,
the kink $x = 3$ is a **non-smooth local minimizer** with $f(3) = \tfrac{3}{2}$. The local
maximizer $x_+ \approx 2.1547$ is therefore a basin boundary. Importantly, fixed-step
subgradient descent started in the **open** interval $(x_+, 3)$ does not settle at the
kink: from the right, $f' \geq 5$ throws iterates back left, producing a persistent
**limit cycle around $x=3$** (demonstrated empirically in §6.5). The endpoint $x_0 = 3$
itself is the measure-zero exception: with the smallest-norm selection $0 \in \partial
f(3)$, it is an exact fixed point of the update. Either way, iterates started past $x_+$
remain *trapped near* the kink without converging to $x_\star$ — which matters for any
SGD path that wanders past $x_+$.

**Interpretation.** The left panel shows the shallow global basin around
$x_\star \approx -0.155$ against the tall wall past the kink; the right panel's sign
changes at $x_-$ and $x_+$ explain every trajectory below — starts in $(x_-, x_+)$
drift left to $x_\star$, starts in $(x_+, 3]$ drift right toward the kink.

In [ ]:
xs = np.linspace(-1.0, 4.0, 1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(xs, f(xs), label="$f(x)$")
axes[0].axvline(3.0, ls="--", c="gray", label="kink $x=3$")
axes[0].plot([X_STAR], [F_STAR], "o", c="#e84855", label="$x_\\star$")
axes[0].set(title="Objective $f$", xlabel="$x$", ylabel="$f(x)$")
axes[0].legend()

mask_l, mask_r = xs < 3.0, xs > 3.0
axes[1].plot(xs[mask_l], fprime(xs[mask_l]), c="#1b998b", label="$f'(x)$ (branches)")
axes[1].plot(xs[mask_r], fprime(xs[mask_r]), c="#1b998b")
axes[1].axvline(3.0, ls="--", c="gray")
axes[1].axhline(0.0, lw=0.8, c="k")
axes[1].plot([X_STAR, X_PLUS], [0, 0], "o", c="#e84855",
             label=f"roots $x_-\\approx{X_STAR:.4f}$, $x_+\\approx{X_PLUS:.4f}$")
axes[1].set(title="Piecewise derivative $f'$", xlabel="$x$", ylabel="$f'(x)$")
axes[1].legend()
fig.tight_layout()
save_fig(fig, "fig1_function_and_derivative")
plt.show()


## 6. Gradient Descent

$$
x_{k+1} = x_k - \eta\, g_k, \qquad
g_k \in \begin{cases}\{f'(x_k)\}, & x_k \neq 3,\\ \partial f(3), & x_k = 3.\end{cases}
$$

For sufficiently small $\eta$ the map is a contraction near $x_\star$
(since $f''(x_\star) = 2\sqrt{3} > 0$), giving linear convergence.

In [ ]:
def gd(x0, eta, steps):
    """Vanilla GD with subgradient selection at the kink. Returns array of iterates."""
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * subgrad(xs[-1]))
    return np.array(xs)

def sgd(x0, steps, eta0, gamma=None, sigma=SIGMA, rng=None):
    """SGD: x_{k+1} = x_k - eta_k (f'(x_k) + eps_k), eps_k ~ N(0, sigma^2).
    gamma=None -> constant step; else eta_k = eta0 / (1 + gamma k)."""
    rng = rng or np.random.default_rng(SEED)
    xs = [float(x0)]
    for k in range(steps):
        eta_k = eta0 if gamma is None else eta0 / (1.0 + gamma * k)
        noise = rng.normal(0.0, sigma)
        xs.append(xs[-1] - eta_k * (subgrad(xs[-1]) + noise))
    return np.array(xs)

def metrics(xs, tol=TOL):
    """final gap, best-so-far gap, steps-to-tolerance (np.nan if never reached)."""
    gaps = f(xs) - F_STAR
    hit = np.nonzero(gaps <= tol)[0]
    return {
        "final_gap": float(gaps[-1]),
        "best_gap": float(gaps.min()),
        "steps_to_tol": int(hit[0]) if hit.size else np.nan,
    }


### 6.1 Local Step Geometry

At $x_0 = 2$: $f'(x_0) = \tfrac12$, so with $\eta = 0.2$ the update is
$x' = x_0 - \eta f'(x_0) = 1.9$. The tangent line
$y = f(x_0) + f'(x_0)(x - x_0)$ shows how the local slope determines direction and length.

In [ ]:
x0_geom = 2.0
slope = fprime(x0_geom)
x_new = x0_geom - ETA_GEOM * slope
assert np.isclose(x_new, 1.9), "step-geometry sanity check (handout value)"

xs_loc = np.linspace(1.2, 2.6, 200)
tangent = f(x0_geom) + slope * (xs_loc - x0_geom)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs_loc, f(xs_loc), label="$f(x)$")
ax.plot(xs_loc, tangent, "--", c="#1b998b", label="tangent at $x_0$")
ax.plot([x0_geom], [f(x0_geom)], "o", c="k", label="$x_0 = 2$")
ax.plot([x_new], [f(x0_geom) + slope * (x_new - x0_geom)], "o", c="#e84855",
        label=f"tangent prediction at $x'={x_new:.1f}$")
ax.plot([x_new], [f(x_new)], "s", c="#5f6caf", ms=6,
        label=f"true $f(x') = {f(x_new):.4f}$")
ax.annotate("", xy=(x_new, f(x0_geom)), xytext=(x0_geom, f(x0_geom)),
            arrowprops=dict(arrowstyle="->", color="#e84855"))
ax.set(title=f"Gradient step at $x_0=2$, $\\eta={ETA_GEOM}$",
       xlabel="$x$", ylabel="$y$")
ax.legend()
save_fig(fig, "fig2_step_geometry")
plt.show()


### 6.2 Three Trajectories, Constant Step

$\eta = 0.15$ from $x_0 \in \{-1.0,\ 0.5,\ 2.0\}$; all should converge to
$x_\star \approx -0.1547$.

**Interpretation.** All three protocol starts reach $x_\star$: $x_0 = -1.0$ descends the
left slope directly, while $x_0 = 0.5$ and $x_0 = 2.0$ ride the interior branch leftward —
the longer the ride, the more iterations (quantified in §6.4).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
xs_plot = np.linspace(-1.2, 3.0, 600)
ax.plot(xs_plot, f(xs_plot), c="0.6", lw=1.2, label="$f(x)$")
for x0, c, z in zip(X0_LIST, ["#1b998b", "#e84855", "#5f6caf"], [3, 5, 4]):
    path = gd(x0, ETA_TRAJ, 60)   # zorder puts x0=0.5 on top (it shares the x0=2.0 corridor)
    ax.plot(path, f(path), "o-", ms=3, c=c, zorder=z, label=f"$x_0={x0}$")
ax.axvline(X_STAR, ls=":", c="k", lw=0.8)
ax.set(title=f"GD iterates, $\\eta={ETA_TRAJ}$", xlabel="$x$", ylabel="$f(x)$")
ax.legend()
save_fig(fig, "fig3_gd_trajectories")
plt.show()


### 6.3 Overshooting and Oscillation

From $x_0=0.5$ with $\eta=0.6$, the handout reports the sequence
$0.5,\ -0.475,\ 0.283,\ -0.454,\ 0.249,\ \dots$ alternating around $x_\star$.
The cell verifies our implementation reproduces those exact values —
a concrete correctness check against the source.

In [ ]:
seq = gd(0.5, ETA_OVERSHOOT, 4)
print("iterates:", np.round(seq, 3))
expected = np.array([0.5, -0.475, 0.283, -0.454, 0.249])
assert np.allclose(np.round(seq, 3), expected, atol=1e-3), \
    "overshoot sequence deviates from handout values"
print("matches handout sequence PASSED")

small = gd(0.5, 0.10, 25)
large = gd(0.5, ETA_OVERSHOOT, 25)
fig, ax = plt.subplots(figsize=(7.5, 4))
xs_loc = np.linspace(-0.8, 0.6, 400)
ax.plot(xs_loc, f(xs_loc), c="0.6", lw=1.2, label="$f(x)$")
ax.plot(small, f(small), "o-", ms=3, label="small $\\eta=0.10$")
ax.plot(large, f(large), "o-", ms=3, label=f"large $\\eta={ETA_OVERSHOOT}$")
ax.axvline(X_STAR, ls=":", c="k", lw=0.8, label="$x_\\star$")
ax.set(title="Overshooting with a large step size", xlabel="$x$", ylabel="$f(x)$")
ax.legend()
save_fig(fig, "fig4_overshoot")
plt.show()


### 6.4 Step-Size Sensitivity, Full Protocol

GD for every $(x_0, \eta)$ pair of the protocol, $K = 200$: final gap per step size,
plus convergence curves. The handout's Fig. 9 is schematic; these are actual results.

**Honest deviation from the handout's schematic Fig. 9:** in the actual runs, GD reaches
the analytic minimum to machine precision for *every* protocol $(x_0, \eta)$ pair, so the
*final gap* does not discriminate between step sizes at $K=200$. The informative metric is
**steps-to-tolerance**, plotted below instead; the convergence curves tell the same story.

**Interpretation.** Steps-to-tolerance falls roughly geometrically as $\eta$ doubles
(from $x_0 = 2.0$: 41/20/13/10), so within the stable range larger steps simply win —
consistent with the handout's suggested useful range $[0.10, 0.20]$ and with the linear
contraction rate quantified by the quadratic baseline (§9).

In [ ]:
results_gd = {}
for x0 in X0_LIST:
    for eta in ETA_GD:
        xs_run = gd(x0, eta, K)
        results_gd[(x0, eta)] = {"path": xs_run, **metrics(xs_run)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
width = 0.25
for i, x0 in enumerate(X0_LIST):
    stt = [results_gd[(x0, eta)]["steps_to_tol"] for eta in ETA_GD]
    axes[0].bar(np.arange(len(ETA_GD)) + (i - 1) * width, stt, width, label=f"$x_0={x0}$")
axes[0].set(title=f"Steps to tolerance ($f(x_k)-f(x_\\star) \\leq$ {TOL})",
            xlabel="step size $\\eta$", ylabel="iterations",
            xticks=range(len(ETA_GD)), xticklabels=[str(e) for e in ETA_GD])
axes[0].legend()

EPS = np.finfo(np.float64).eps
for eta in ETA_GD:
    gaps = f(results_gd[(0.5, eta)]["path"]) - F_STAR
    axes[1].semilogy(np.maximum(gaps, EPS), label=f"$\\eta={eta}$")
axes[1].set(title="Convergence from $x_0=0.5$", xlabel="iteration $k$",
            ylabel="$f(x_k)-f(x_\\star)$")
axes[1].legend()
fig.tight_layout()
save_fig(fig, "fig5_stepsize_sensitivity")
plt.show()


### 6.5 Kink-Basin Probe: Fixed-Step GD Trapped in a Limit Cycle

Deterministic GD from starts inside the open kink basin $(x_+, 3)$ and beyond
($x_0 > 3$), run for the full horizon with **no early stopping**. The prediction from §5:
no convergence to tolerance, iterates cycling in a band around $x = 3$ (while the
endpoint $x_0 = 3$ itself would sit still, being an exact fixed point).

**Interpretation:** the observed late-stage bands (printed below) confirm the limit
cycle — the trajectories neither reach $x_\star$ nor settle at the kink itself. This is
the deterministic skeleton of the SGD trapping seen in §7.

## 7. Stochastic Gradient Descent

$$
x_{k+1} = x_k - \eta_k\big(f'(x_k) + \varepsilon_k\big), \qquad
\mathbb{E}[\varepsilon_k \mid x_k] = 0,\quad \mathbb{E}[\varepsilon_k^2 \mid x_k] \le \sigma^2 .
$$

Constant steps produce a noise floor: the linearized steady-state variance around
$x_\star$ (handout, Prop. 2, with $\mu = f''(x_\star) = 2\sqrt{3}$) is

$$
\lim_{k\to\infty} \mathbb{E}\,y_k^2 \;=\; \frac{\eta\sigma^2}{2\mu - \eta\mu^2}
\;\approx\; \frac{\eta\sigma^2}{2\mu}\quad (\eta \ll 1),
$$

so RMS distance scales like $\sqrt{\eta}\,\sigma/\sqrt{2\mu}$.
Diminishing schedules $\eta_k = \eta_0/(1+\gamma k)$ (Robbins–Monro) remove the floor at
the cost of slower initial progress.

### 7.1 Sample Paths

**Disclosure:** $x_0 = 2.2$ is the handout *Figure 7* illustration value, used here only to
replicate that figure; it lies inside the kink basin ($> x_+$), so with $\eta = 0.12$ and
$\sigma = 0.5$ the paths bounce against the kink's steep right wall rather than reach
$x_\star$ — an honest, real-run deviation from the schematic. The *protocol* experiments
in §7.2 use the prescribed starts $x_0 \in \{-1.0, 0.5, 2.0\}$.

In [ ]:
basin_starts = [2.5, 2.8, 3.5]
fig, ax = plt.subplots(figsize=(8, 4))
for x0 in basin_starts:
    path = gd(x0, 0.15, K)
    tail = path[-50:]
    hit = np.nonzero(f(path) - F_STAR <= TOL)[0]
    print(f"x0 = {x0}: reached tol = {bool(hit.size)}, "
          f"late-stage band = [{tail.min():.3f}, {tail.max():.3f}]")
    ax.plot(path, lw=1.0, label=f"$x_0 = {x0}$")
ax.axhline(3.0, ls="--", c="gray", lw=0.9, label="kink $x=3$")
ax.axhline(X_STAR, ls=":", c="k", lw=0.9, label="$x_\\star$")
ax.set(title="Fixed-step GD ($\\eta=0.15$) trapped in a limit cycle around the kink",
       xlabel="iteration $k$", ylabel="$x_k$")
ax.legend(fontsize=8)
save_fig(fig, "fig6_kink_limit_cycle")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
for i in range(3):
    path = sgd(2.2, 40, eta0=0.12, rng=np.random.default_rng(SEED + i))
    ax.plot(path, "o-", ms=2.5, label=f"path {chr(65 + i)} (seed {SEED + i})")
ax.axhline(X_STAR, ls=":", c="k", lw=0.9, label="$x_\\star$")
ax.set(title=f"SGD sample paths, constant $\\eta=0.12$, $\\sigma={SIGMA}$",
       xlabel="iteration $k$", ylabel="$x_k$")
ax.legend()
save_fig(fig, "fig7_sgd_paths")
plt.show()


### 7.2 Constant vs Diminishing Schedules

Replicates handout Fig. 8 with real protocol runs. Full Section 7 protocol: starts $x_0 \in \{-1.0, 0.5, 2.0\}$, constant
$\eta_0 \in \{0.2, 0.1\}$ and diminishing $\eta_k = \eta_0/(1+\gamma k)$ with
$\gamma \in \{0.02, 0.05\}$, $K = 200$, **20 seeds per configuration**. The figure shows
the representative start $x_0 = 0.5$ as **median curves with interquartile bands**
(clipped at machine epsilon for the log scale — disclosed); the paired statistics and
metrics table below cover all three starts.

**Basin effect:** $x_0 = 2.0$ sits only $\approx 0.15$ below the basin boundary
$x_+ \approx 2.1547$, so noise can kick individual paths over $x_+$ into the kink's limit
cycle (§6.5). The escape/trap fractions are reported explicitly below.

In [ ]:
N_REPS = 20                      # seeds per configuration
SGD_SEEDS = [SEED + 100 + r for r in range(N_REPS)]   # recorded
EPS64 = np.finfo(np.float64).eps

def gap_matrix(x0, eta0, gamma):
    """(N_REPS, K+1) matrix of objective gaps for one configuration."""
    rows = [f(sgd(x0, K, eta0=eta0, gamma=gamma, rng=np.random.default_rng(s))) - F_STAR
            for s in SGD_SEEDS]
    return np.vstack(rows)

# Full protocol grid: 3 starts x 2 eta0 x {const, 0.02, 0.05}
results_sgd = {}
for x0 in X0_LIST:
    for eta0 in ETA0_SGD:
        for gamma in [None] + GAMMA_SGD:
            results_sgd[(x0, eta0, gamma)] = gap_matrix(x0, eta0, gamma)

# Figure: representative start x0 = 0.5, median with IQR bands
X0_FIG = 0.5
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for eta0, base_c in zip(ETA0_SGD, ["#1b998b", "#5f6caf"]):
    for gamma, ls in [(None, "-"), (0.02, "--"), (0.05, ":")]:
        G = np.maximum(results_sgd[(X0_FIG, eta0, gamma)], EPS64)
        med = np.median(G, axis=0)
        q1, q3 = np.percentile(G, [25, 75], axis=0)
        lab = (f"constant $\\eta={eta0}$" if gamma is None
               else f"$\\eta_k={eta0}/(1+{gamma}k)$")
        ax.semilogy(med, ls, c=base_c, label=lab)
        ax.fill_between(range(K + 1), q1, q3, color=base_c, alpha=0.12)
ax.set(title=f"Objective gap from $x_0={X0_FIG}$: median of {N_REPS} seeds, IQR shaded "
             f"(gaps clipped at machine eps for log scale)",
       xlabel="iteration $k$", ylabel="$f(x_k)-f(x_\\star)$")
ax.legend(fontsize=8, loc="upper right")
save_fig(fig, "fig8_sgd_schedules")
plt.show()

# Paired statistics: diminishing vs constant (same eta0, same seed), final gaps
print("Paired final-gap comparison (diminishing vs constant, same seed):")
for x0 in X0_LIST:
    for eta0 in ETA0_SGD:
        fc = results_sgd[(x0, eta0, None)][:, -1]
        for gamma in GAMMA_SGD:
            fd = results_sgd[(x0, eta0, gamma)][:, -1]
            wins = int((fd < fc).sum())
            ratio = float(np.median(fd / np.maximum(fc, EPS64)))
            print(f"  x0={x0:5}, eta0={eta0}, gamma={gamma}: "
                  f"diminishing wins {wins}/{N_REPS}, median ratio {ratio:.3f}")

# Basin escape statistics for x0 = 2.0 (final iterate left of x_+ = escaped)
print("\nKink-basin trap fractions (x0 = 2.0): fraction of seeds ending with x_K > x_+")
for eta0 in ETA0_SGD:
    for gamma in [None] + GAMMA_SGD:
        xK = np.array([sgd(2.0, K, eta0=eta0, gamma=gamma,
                           rng=np.random.default_rng(s))[-1] for s in SGD_SEEDS])
        trapped = int((xK > X_PLUS).sum())
        gname = "const" if gamma is None else f"g={gamma}"
        print(f"  eta0={eta0}, {gname}: trapped {trapped}/{N_REPS}")


## 8. Metrics Summary

GD rows are single deterministic runs; SGD rows report the **median-over-20-seeds** gap
curve (final, best, and steps for the median curve to reach tol $=10^{-3}$).

In [ ]:
header = f"{'method':<26}{'x0':>6}{'eta':>12}{'final gap':>12}{'best gap':>12}{'steps<tol':>11}"
print(header); print("-" * len(header))
for (x0, eta), res in results_gd.items():
    print(f"{'GD constant':<26}{x0:>6}{eta:>12}{res['final_gap']:>12.2e}"
          f"{res['best_gap']:>12.2e}{str(res['steps_to_tol']):>11}")
for (x0, eta0, gamma), G in results_sgd.items():
    label = "SGD constant" if gamma is None else f"SGD diminishing g={gamma}"
    med = np.median(G, axis=0)                      # median gap curve over seeds
    hit = np.nonzero(med <= TOL)[0]
    stt = int(hit[0]) if hit.size else "n/a"
    print(f"{label:<26}{x0:>6}{eta0:>12}{med[-1]:>12.2e}"
          f"{med.min():>12.2e}{str(stt):>11}")


## 9. Quadratic Baseline — Stability Reference

The coach guide's scope includes GD on a quadratic; this section covers it while
calibrating the thresholds used above. On $q(x) = \tfrac12 a x^2$ with $a = f''(x_\star) = 2\sqrt{3}$ (matching the local
curvature at the minimizer), GD contracts exactly by $|1 - \eta a|$ per step. This
calibrates the step-size thresholds quoted throughout: stability requires
$\eta < 2/a \approx 0.577$ near $x_\star$, while the kink's right branch has
$f''(3^+) = 6$, giving the tighter local bound $2/6 \approx 0.333$. The cell verifies the
empirical contraction matches theory and shows divergence just past $2/a$.

In [ ]:
a = 2.0 * np.sqrt(3.0)                       # local curvature at x_star
def q(x): return 0.5 * a * x**2
def qgrad(x): return a * x

print(f"stability threshold 2/a = {2/a:.4f}   (right-branch-at-kink bound 2/6 = {2/6:.4f})")
for eta in [0.10, 0.20, 0.50, 0.60]:
    x = 1.0
    ratios = []
    for _ in range(30):
        x_new = x - eta * qgrad(x)
        if abs(x) > 0:
            ratios.append(abs(x_new) / abs(x))
        x = x_new
        if abs(x) > 1e6:
            break
    emp = max(ratios)
    theory = abs(1 - eta * a)
    status = "stable" if eta < 2 / a else "DIVERGES"
    print(f"eta={eta:.2f}: |1-eta*a|={theory:.3f}  empirical max ratio={emp:.3f}  -> {status}")
    if eta < 2 / a:
        assert np.isclose(emp, theory, atol=1e-9), "contraction mismatch"
print("quadratic contraction check PASSED")

## 10. Interpretation Notes

- **Calculus → algorithm.** The sign structure of $f'$ funnels all three protocol starts
  to $x_\star$: every GD run reaches the $10^{-3}$ tolerance in 2–41 steps and machine
  precision ($\sim 10^{-16}$) well before $K = 200$.
- **Step sizes.** All four protocol $\eta$ are stable — consistent with the local bound
  $2/f''(x_\star) \approx 0.577$ (§9) — and speed ranks monotonically with $\eta$
  (from $x_0 = 2.0$: 41/20/13/10 steps), matching the handout's useful range
  $[0.10, 0.20]$. The overshoot demo's $\eta = 0.6$ sits past the bound, where the
  quadratic baseline confirms divergence ($|1-\eta a| = 1.078$).
- **SGD noise levels match Prop. 2.** Median final gaps at $K = 200$ (20 seeds) for
  constant steps: $2.14\times10^{-2}$ ($\eta = 0.2$) and $7.79\times10^{-3}$
  ($\eta = 0.1$), consistent with the linearized steady-state predictions
  $1.91\times10^{-2}$ and $7.56\times10^{-3}$ to within ~12% and ~3%.
- **Constant vs diminishing.** Diminishing achieves lower final gaps **by $K = 200$** in
  16–19 of 20 paired seeds (median ratios 0.044–0.142), trading slower early progress;
  the median is not monotone, so this is not a claim of steady convergence.
- **Coupling.** SGD final gaps are identical across starts by design: shared per-seed
  noise plus contraction makes trajectories couple; only transients differ. Expected,
  not a copy-paste artifact.
- **Kink trap — real but transient here.** Deterministic GD from $\{2.5, 2.8, 3.5\}$
  cycles forever in $[2.261, 3.236]$ (§6.5); yet under protocol noise 0/20 seeds from
  $x_0 = 2.0$ end trapped — the cycle's left edge sits only ~0.11 above $x_+$, so noise
  re-escapes easily. Trapping shapes transients, not endpoints, at these settings.
- **Disclosed deviations from the handout's schematics.** GD gaps saturate at machine
  precision (so §6.4 reports steps-to-tolerance); the Fig. 7 replication ($x_0 = 2.2$,
  inside the kink basin) honestly bounces off the kink wall instead of approaching
  $x_\star$.

## 11. Limitations & Extensions

Not implemented (handout §9 marks these as extensions, not requirements): heavy-ball /
Nesterov comparison, Adam, contraction-bound plot (Fig. 10), proximal step for
$f(x)+\lambda|x|$, multi-dimensional separable variants, and smoothed
$\sqrt{g^2+\delta^2}$ approximations. Kink handling uses the smallest-norm subgradient;
alternatives (backtracking across the kink) are noted in handout §8.

## 12. Reproducibility Footer

In [ ]:
import platform, matplotlib, time
print(f"python     : {platform.python_version()}")
print(f"numpy      : {np.__version__}")
print(f"matplotlib : {matplotlib.__version__}")
print(f"seed       : {SEED}  (SGD reps use SEED+100..SEED+{100 + N_REPS - 1})")
print(f"figures    : {sorted(p.name for p in ASSETS.glob('*.png'))}")
